# Glue Interactive Sessions - Processamento Batch (Full Load)

Este notebook conecta em uma **sessão interativa do AWS Glue** (conta `331504768406`, região `us-east-1`) e executa os **mesmos passos da classe `Processor`** (`processor.run` / `main._run_batch`) em modo **batch**: leitura do full-load Parquet no landing, validação de qualidade, escrita de rejeitados e MERGE Delta no raw.

**Pré-requisitos:**

- Kernel **Glue PySpark** local: `pip install jupyter boto3 aws-glue-sessions` e depois `install-glue-kernels`. Abra com `jupyter notebook` e escolha o kernel `Glue PySpark`.
- Credenciais AWS válidas com acesso ao Glue e S3 (usuário `lake-admin` / grupo `datalake-admins`).
- Role `role-datalake-analytics` com as permissões de interactive sessions (já provisionadas via Terraform em `infra/iam.tf`).
- `helpers.zip` publicado no bucket workspace (`aws-glue/jobs/flight-radar/src/dependencies/helpers.zip`).

> **Delta Lake**: a célula `%%configure` abaixo é obrigatória - o `writer.py` do projeto importa `delta.tables`. Sem ela a importação falha com `ModuleNotFoundError: No module named 'delta'`.


In [ ]:
# 1) Configuração da sessão (magics do kernel AWS Glue)
# A célula seguinte (%%configure) habilita o Delta Lake. Cell magics
# (%%configure/%%tags) devem ser a primeira linha da célula, sem comentários.
%glue_version 5.0
%iam_role arn:aws:iam::331504768406:role/role-datalake-analytics
%region us-east-1
%worker_type G.1X
%number_of_workers 2
%idle_timeout 30
%session_id_prefix flight-radar-batch

In [ ]:
%%configure
{
  "--datalake-formats": "delta",
  "--conf": "spark.sql.extensions=io.delta.sql.DeltaSparkSessionExtension --conf spark.sql.catalog.spark_catalog=org.apache.spark.sql.delta.catalog.DeltaCatalog --conf spark.delta.logStore.class=org.apache.spark.sql.delta.storage.S3SingleDriverLogStore"
}

In [ ]:
%%tags
{"Environment": "production", "Project": "flight-radar-glue", "Mode": "batch"}

In [ ]:
# Dependências do projeto (helpers.zip já publicado no workspace)
%extra_py_files s3://lakehouse-workspace-331504768406/aws-glue/jobs/flight-radar/src/dependencies/helpers.zip

In [ ]:
# Instanciar a SparkSession / GlueContext
# Em sessões interativas o Spark já existe; getOrCreate retorna a mesma sessão.
from pyspark.context import SparkContext
from awsglue.context import GlueContext

sc = SparkContext.getOrCreate()
glue_context = GlueContext(sc)
spark = glue_context.spark_session

print('Spark version:', spark.version)

In [ ]:
# Dependências do projeto carregadas via %extra_py_files (helpers.zip)
# O zip contém o pacote `src` (src.dependencies.*).
import boto3
from src.dependencies.processor import Processor
from src.dependencies.config_models import Config

account_id = boto3.client("sts").get_caller_identity()["Account"]
config = Config.from_s3(f"s3://lakehouse-workspace-{account_id}/aws-glue/jobs/flight-radar/src/dependencies/config/config.json")

for s in config.sources:
    print(f"{s.order:>2}  {s.source:<20} -> {s.target.database}.{s.target.table}")

## Passos equivalentes ao `Processor.run(..., mode='batch')`

O `main._run_batch` executa, por tabela: (1) leitura via `reader.read(mode='batch')`, (2) `processor.run(source, target, mode='batch', dataframe=raw_df)` - que internamente faz validação de qualidade, escrita de rejeitados e MERGE Delta. Abaixo cada etapa é executada em uma célula separada.

In [ ]:
# Escolher a tabela a testar (aqui: aircraft, que tem dados no landing)
processor = Processor(spark)

source = config.get_source('aircraft')
target = source.target
print(source.source, '->', f'{target.database}.{target.table}')

In [ ]:
# Passo 1 - Leitura batch do full-load Parquet (reader.read com mode='batch')
raw_df = processor._reader.read(source, mode='batch')
print('Linhas lidas:', raw_df.count())
raw_df.printSchema()

In [ ]:
raw_df.show(15, truncate=False)

In [ ]:
# Passo 2 - Validação de qualidade de dados (DataQuality.validate)
valid_df, rejects_df = processor._data_quality.validate(raw_df, target, source)
print('Válidas:', valid_df.count())
print('Rejeitadas:', 0 if rejects_df.isEmpty() else rejects_df.count())

In [ ]:
# Passo 3 - Escrita dos rejeitados (writer.write_rejects)
processor._writer.write_rejects(rejects_df, target)
print('Rejects escritos em:', target.rejected_location)

In [ ]:
# Passo 4 - Escrita Delta com MERGE (writer.write)
# Gera cod_unico, deriva event_date, projeta as colunas do alvo e aplica
# WHEN NOT MATCHED AND Op <> 'D' INSERT / WHEN MATCHED AND Op='D' DELETE / UPDATE ALL.
processor._writer.write(valid_df, target, source)
print('MERGE Delta concluído para', f'{target.database}.{target.table}')

In [ ]:
# Verificação - ler a tabela Delta final (via path, independe do Catálogo)
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, target.location)
print('Colunas:', dt.toDF().columns)
print('Linhas na tabela:', dt.toDF().count())
dt.toDF().show(5)

## Rodar o pipeline completo (todas as tabelas)

Espelha o `main._run_batch`: processa cada tabela em sequência, continuando mesmo se uma falhar.

In [ ]:
# Pipeline completo em batch - equivalente ao main._run_batch
for idx, s in enumerate(config.sources, start=1):
    print(f'[{idx}/{len(config.sources)}] {s.source}')
    try:
        df = processor._reader.read(s, mode='batch')
        processor.run(s, s.target, mode='batch', dataframe=df)
        print('  OK')
    except Exception as exc:
        print('  FAILED:', exc)

In [ ]:
# Status da sessão (mostra tags, role, workers, região)
%status

In [ ]:
# Encerrar a sessão quando terminar
%stop_session